In [1]:
import sys
import os

# Add the project root dynamically. Required by Jupyter
sys.path.append(os.path.abspath(".."))  # adjust path as needed

In [2]:
import pandas as pd

In [3]:
from evaluations import report_metrics
from pipeline import train_pipeline

### Load data

In [4]:
# You may need to adjust these data file paths
train_genos_df = pd.read_csv(f"../data/sample_train_genotype.csv")
train_phenos_df = pd.read_csv(f"../data/sample_train_phenotype.csv")
test_genos_df = pd.read_csv(f"../data/sample_test_genotype.csv")
test_phenos_df = pd.read_csv(f"../data/sample_test_phenotype.csv")

In [5]:
X_train = train_genos_df.iloc[:,1:]
y_train = train_phenos_df.iloc[:, 1].squeeze()
X_test = test_genos_df.iloc[:,1:]
y_test = test_phenos_df.iloc[:, 1].squeeze()

### Test training pipeline

In [6]:
preprocess_params = {
    "imputation-strategy": "mean",
    "imputation-fill-value": 0,
}

#### Simple train-predict

In [13]:
# Simple train-prediction
reducer_name = "LASSOFS"
reducer_params = {"alpha": 0.1, "test_size": 0.2, "n_reps": 200, "r": 0.2}

model_name = "BayesLASSO"
model_params = {}

In [14]:
pipeline = train_pipeline(
    X_train=X_train, y_train=y_train,
    preprocess_params=preprocess_params,
    reducer_name=reducer_name,
    reducer_params=reducer_params,
    model_name=model_name,
    model_params=model_params
)

In [15]:
y_train_pred = pipeline.predict(X_train)
train_metrics = report_metrics(y_train, y_train_pred)
train_metrics

{"Pearson's r": np.float64(0.971870826291965),
 'Top 25% HR': 0.8,
 'Low 25% HR': 0.86}

In [16]:
y_test_pred = pipeline.predict(X_test)
test_metrics = report_metrics(y_test, y_test_pred)
test_metrics

{"Pearson's r": np.float64(0.10835877452426221),
 'Top 25% HR': 0.5,
 'Low 25% HR': 0.0}

#### Grid Search for one reducer and one regressor

In [17]:
reducer_name = "LASSOFS"
reducer_params = {"alpha": 0.1, "test_size": 0.2, "n_reps": 200, "r": 0.2}
reducer_param_grid = {
    'LASSOFS__r': [0.1, 0.25],
    'LASSOFS__test_size': [0.2, 0.4]
}

model_name = "BayesLASSO"
model_params = {}
model_param_grid = {}

In [18]:
pipeline = train_pipeline(
    X_train=X_train, y_train=y_train,
    preprocess_params=preprocess_params,
    reducer_name=reducer_name,
    reducer_params=reducer_params,
    model_name=model_name,
    model_params=model_params,
    reducer_param_grid=reducer_param_grid,
    model_param_grid=model_param_grid,
    gridsearch_cv_folds=5,
    run_gridsearch=True
)

Fitting 5 folds for each of 4 candidates, totalling 20 fits
[CV 1/5] END LASSOFS__r=0.1, LASSOFS__test_size=0.2;, score=0.113 total time=  28.0s
[CV 2/5] END LASSOFS__r=0.1, LASSOFS__test_size=0.2;, score=0.306 total time=  23.2s
[CV 3/5] END LASSOFS__r=0.1, LASSOFS__test_size=0.2;, score=0.442 total time=  22.4s
[CV 4/5] END LASSOFS__r=0.1, LASSOFS__test_size=0.2;, score=-0.062 total time=  24.6s
[CV 5/5] END LASSOFS__r=0.1, LASSOFS__test_size=0.2;, score=-0.013 total time=  21.8s
[CV 1/5] END LASSOFS__r=0.1, LASSOFS__test_size=0.4;, score=0.109 total time=  24.7s
[CV 2/5] END LASSOFS__r=0.1, LASSOFS__test_size=0.4;, score=0.285 total time=  22.3s
[CV 3/5] END LASSOFS__r=0.1, LASSOFS__test_size=0.4;, score=0.461 total time=  21.9s
[CV 4/5] END LASSOFS__r=0.1, LASSOFS__test_size=0.4;, score=-0.040 total time=  23.7s
[CV 5/5] END LASSOFS__r=0.1, LASSOFS__test_size=0.4;, score=0.020 total time=  21.0s
[CV 1/5] END LASSOFS__r=0.25, LASSOFS__test_size=0.2;, score=0.119 total time=  27.5s
[

In [19]:
y_train_pred = pipeline.predict(X_train)
train_metrics = report_metrics(y_train, y_train_pred)
train_metrics

{"Pearson's r": np.float64(0.9598714624177233),
 'Top 25% HR': 0.82,
 'Low 25% HR': 0.86}

In [20]:
y_test_pred = pipeline.predict(X_test)
test_metrics = report_metrics(y_test, y_test_pred)
test_metrics

{"Pearson's r": np.float64(0.15035374638835622),
 'Top 25% HR': 0.5,
 'Low 25% HR': 0.0}

#### Grid search for multiple reducer + regressor combinations